# Unit 7 — Network Programming
**Course:** BT151CO — Object-Oriented Programming  
**Unit:** 7 of 8  
**Duration:** 4 Hours  

---

## What Is This Notebook?

This notebook teaches you how to write **client-server programs** in Python using the built-in `socket` module. No installation is required — sockets are part of the Python standard library.

> **Important:** A server and its client cannot run in the **same** notebook cell because the server blocks while waiting for connections. This notebook uses a **background thread trick** to let both run inside Jupyter for demonstration. For real projects you always run server and client in **two separate terminals**.

---

## Learning Objectives

By the end of this unit you will be able to:

1. Explain what a socket is and describe its two key arguments
2. Describe the difference between a client and a server
3. Explain TCP vs UDP and choose the right one
4. Write a complete TCP client program
5. Write a complete TCP server program that accepts connections in a loop
6. Use threads to handle multiple clients simultaneously
7. Handle network errors gracefully with `try / except`
8. Combine sockets with SQLite to build a data-driven server


## Table of Contents

1. [Introduction to Network Programming](#s1)
2. [A Daytime Server Example](#s2)
3. [Clients and Servers](#s3)
4. [Writing the Client Program](#s4)
5. [Writing the Server Program](#s5)
6. [Library Server — Capstone (Sockets + SQLite)](#s6)
7. [Practice Tasks](#practice)
8. [Debugging Exercises](#debug)
9. [Mini Coding Challenges](#challenges)
10. [Summary & Checklist](#summary)


<a id='s1'></a>
---
## Section 1 — Introduction to Network Programming

### What Is a Socket?

A **socket** is a Python object that represents **one endpoint** of a two-way communication channel between two programs — they can be on the same machine or on opposite sides of the world.

| Concept | Meaning | Analogy |
|---|---|---|
| **IP Address** | Unique number identifying a machine | Postal address |
| **Port** | Number identifying a service on that machine | Flat number |
| **Protocol** | Agreed rules for communication | A shared language |
| **Socket** | One end of the communication pipe | A phone handset |
| **Client** | Program that initiates the connection | Customer in a shop |
| **Server** | Program that waits and responds | The shopkeeper |

### TCP vs UDP

| Feature | TCP | UDP |
|---|---|---|
| Reliability | Guaranteed | Fire-and-forget |
| Order | Arrives in order | May arrive out of order |
| Speed | Slightly slower | Very fast |
| Use cases | Web, email, chat | Video calls, games, DNS |

> **We use TCP throughout this unit** — it is the reliable choice for most client-server applications.


In [ ]:
# ── Imports ──────────────────────────────────────────────────────
import socket
import threading
import time
import datetime
import json
import sqlite3

print('All imports OK')
print(f'Python socket version: {socket.__doc__[:40]}...')


In [ ]:
# ── 1.1 Creating a Socket Object ─────────────────────────────────
# socket.AF_INET     = use IPv4 (standard internet addresses)
# socket.SOCK_STREAM = use TCP (reliable, ordered)

sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
print(f'Socket created: {sock}')
print(f'Family  : {sock.family}  (AF_INET = {socket.AF_INET})')
print(f'Type    : {sock.type}    (SOCK_STREAM = {socket.SOCK_STREAM})')
sock.close()   # always close when done
print('Socket closed.')


### The `with` Statement — Always Use It

Just like files, sockets must be **closed** after use. The `with` statement guarantees this — even if an exception occurs.

```python
# BAD — socket might not be closed if an error occurs
sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
sock.connect(('localhost', 65432))
sock.close()   # never reached if connect() raises!

# GOOD — with guarantees cleanup
with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
    sock.connect(('localhost', 65432))
    # sock.close() called automatically
```


In [ ]:
# ── 1.2 Inspecting Socket Constants ──────────────────────────────
constants = [
    ('AF_INET',      socket.AF_INET),
    ('AF_INET6',     socket.AF_INET6),
    ('SOCK_STREAM',  socket.SOCK_STREAM),
    ('SOCK_DGRAM',   socket.SOCK_DGRAM),
    ('SOL_SOCKET',   socket.SOL_SOCKET),
    ('SO_REUSEADDR', socket.SO_REUSEADDR),
]

print(f'{'Constant':<20} {'Value':>6}')
print('-' * 28)
for name, val in constants:
    print(f'{name:<20} {val:>6}')


In [ ]:
# ── 1.3 localhost and Port Numbers ───────────────────────────────
# 'localhost' and '127.0.0.1' are identical — the loopback address.
# Data sent here never leaves your machine — perfect for development.

for name in ('localhost', '127.0.0.1'):
    resolved = socket.gethostbyname(name)
    print(f'{name:<15} resolves to  {resolved}')

print()
print('Safe port range for your programs: 10000 – 65000')
print('Well-known ports (0-1023) need admin rights — avoid them.')


<a id='s2'></a>
---
## Section 2 — A Daytime Server Example

The **Daytime Server** is the simplest useful network service:
1. Client connects
2. Server sends back the current date and time
3. Connection closes

Based on real internet standard RFC 867 (1983).

### The Six-Step Server Pattern

| Step | Call | Purpose |
|---|---|---|
| 1 | `socket.socket(...)` | Create the socket |
| 2 | `setsockopt(SO_REUSEADDR, 1)` | Allow instant port reuse on restart |
| 3 | `bind((HOST, PORT))` | Attach to address and port |
| 4 | `listen(5)` | Start accepting connections |
| 5 | `accept()` | Block — wait for a client |
| 6 | `sendall(data)` | Send response bytes |

> Steps 1–4 are **setup** (run once). Steps 5–6 repeat in `while True:`.


### The Code — View Only

The cells below show the complete server and client code. They are **view-only** (comment style) because a server blocks the notebook. The live demo cell further below uses a background thread.

**`daytime_server.py` — Run in Terminal 1:**
```python
import socket, datetime

HOST, PORT = 'localhost', 17000

with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as srv:
    srv.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)  # Step 2
    srv.bind((HOST, PORT))                                       # Step 3
    srv.listen(5)                                               # Step 4
    print(f'Server running on {HOST}:{PORT}')
    while True:
        client_sock, addr = srv.accept()                        # Step 5
        with client_sock:
            now = datetime.datetime.now()
            msg = now.strftime('%A, %d %B %Y  %H:%M:%S\n')
            client_sock.sendall(msg.encode('utf-8'))            # Step 6
            print(f'Sent time to {addr}')
```

**`daytime_client.py` — Run in Terminal 2:**
```python
import socket

HOST, PORT = 'localhost', 17000

with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
    sock.connect((HOST, PORT))          # Step 1
    data = sock.recv(1024)              # Step 2
    print(data.decode('utf-8').strip()) # Step 3
```


In [ ]:
# ── 2.1 Live Demo — Daytime Server (background thread) ───────────
# This cell starts a tiny daytime server in a background daemon thread
# so the client code in the NEXT cell can connect to it.

DEMO_PORT_DAYTIME = 20100

def _daytime_server():
    """Run a one-shot daytime server in a background thread."""
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as srv:
        srv.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
        srv.bind(('localhost', DEMO_PORT_DAYTIME))
        srv.listen(1)
        srv.settimeout(10)  # auto-exit after 10 s if no client
        try:
            client_sock, addr = srv.accept()
            with client_sock:
                now = datetime.datetime.now()
                msg = now.strftime('%A, %d %B %Y  %H:%M:%S\n')
                client_sock.sendall(msg.encode('utf-8'))
        except socket.timeout:
            pass  # no client connected within 10 s

t = threading.Thread(target=_daytime_server, daemon=True)
t.start()
time.sleep(0.1)  # give the server thread a moment to bind
print(f'Daytime server started on localhost:{DEMO_PORT_DAYTIME}')
print('Run the next cell to connect the client.')


In [ ]:
# ── 2.2 Daytime Client ───────────────────────────────────────────
# Run this cell AFTER cell 2.1

with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
    sock.settimeout(5)
    sock.connect(('localhost', DEMO_PORT_DAYTIME))
    data = sock.recv(1024)
    print('Server says:', data.decode('utf-8').strip())


<a id='s3'></a>
---
## Section 3 — Clients and Servers

### The Client-Server Model

| Property | Client | Server |
|---|---|---|
| **Who starts?** | Client always initiates | Server waits passively |
| **How many?** | One per connection | One server → many clients |
| **Port** | Random (assigned by OS) | Fixed, well-known port |
| **Lifecycle** | Short — one task, then done | Long — runs continuously |
| **Pattern** | `connect → send → recv` | `bind → listen → accept → loop` |

### The TCP Three-Way Handshake

```
Client                    Server
  |                          |
  |── SYN ─────────────────▶|   'Can we connect?'
  |◀── SYN-ACK ──────────────|   'Yes, I am ready.'
  |── ACK ─────────────────▶|   'Great, let us go.'
  |                          |
  |══ DATA FLOWS BOTH WAYS ══|
  |                          |
  |── FIN ─────────────────▶|   'I am done.'
  |◀── FIN-ACK ──────────────|   'Acknowledged.'
```

> Python handles the handshake automatically. You just call `connect()` on the client and `accept()` on the server.

### Two Socket Objects After `accept()`

After calling `accept()`, the server has **two** sockets:

| Socket | Role |
|---|---|
| `server_socket` | Keeps listening — **never** used for data |
| `client_socket` | Dedicated to this one client — all data here |

> **Never** send data through `server_socket` — always use the `client_socket` returned by `accept()`.


In [ ]:
# ── 3.1 Getting Machine Information ─────────────────────────────
hostname = socket.gethostname()
local_ip  = socket.gethostbyname(hostname)

print(f'Hostname  : {hostname}')
print(f'Local IP  : {local_ip}')
print(f'Loopback  : 127.0.0.1 (always available)')

# Port ranges
print()
print('Port ranges:')
print('  0   – 1 023  : Well-known ports (need admin rights)')
print('  1024 – 49151 : Registered application ports')
print('  10000– 65000 : Safe range for your own programs')


In [ ]:
# ── 3.2 Check if a Port Is in Use ───────────────────────────────
def is_port_open(host, port, timeout=1):
    """Try to connect; return True if something is listening there."""
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.settimeout(timeout)
        try:
            s.connect((host, port))
            return True
        except (ConnectionRefusedError, socket.timeout, OSError):
            return False

# Test a few well-known ports
for port in [80, 443, 65432, 20100]:
    status = 'OPEN' if is_port_open('localhost', port) else 'closed'
    print(f'localhost:{port:<6} -> {status}')


<a id='s4'></a>
---
## Section 4 — Writing the Client Program

### The Client Template

```python
import socket

HOST = 'localhost'
PORT = 65432

with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
    sock.settimeout(5)              # don't hang forever
    sock.connect((HOST, PORT))      # reach out to the server
    sock.sendall(b'Hello!')         # send bytes
    data = sock.recv(1024)          # receive up to 1024 bytes
    print(data.decode('utf-8'))     # bytes -> string
```

### Bytes vs Strings — The Critical Rule

> **Sockets transmit BYTES, not strings.** You must always convert.

| Direction | Operation | Code |
|---|---|---|
| String → Bytes (to send) | **Encode** | `'hello'.encode('utf-8')` |
| Bytes → String (to display) | **Decode** | `data.decode('utf-8')` |


In [ ]:
# ── 4.1 Bytes vs Strings ────────────────────────────────────────
msg = 'Hello, Network!'

# Encoding: str -> bytes (ready to send over the socket)
as_bytes = msg.encode('utf-8')
print(f'Original string : {msg!r}   type={type(msg).__name__}')
print(f'Encoded bytes   : {as_bytes!r}   type={type(as_bytes).__name__}')

# Decoding: bytes -> str (after receiving from the socket)
back_to_str = as_bytes.decode('utf-8')
print(f'Decoded back    : {back_to_str!r}   type={type(back_to_str).__name__}')

print()
# Bytes literals use a b prefix
b1 = b'Hello'         # bytes literal
b2 = 'Hello'.encode() # same result
print(f'b1 == b2 : {b1 == b2}')
print(f'bool(b"") : {bool(b"")}  <-- empty bytes is FALSY')


In [ ]:
# ── 4.2 sendall() vs send() ─────────────────────────────────────
# send()    - may send fewer bytes than requested
# sendall() - loops until all bytes are sent; always use this

# We cannot call send() without a live connection,
# but we can illustrate the logic:

data = b'A' * 100   # 100 bytes of data

print(f'Data length: {len(data)} bytes')
print()
print('send()    -> may return fewer than', len(data), 'bytes sent')
print('           -> you MUST check the return value and loop')
print()
print('sendall() -> returns None; raises OSError on failure')
print('           -> always use sendall() for complete messages')


In [ ]:
# ── 4.3 Error Handling in the Client ────────────────────────────
# Demonstrate each exception by trying to connect to a closed port.

def safe_connect(host, port):
    """Try to connect; show what exception is raised if it fails."""
    try:
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            s.settimeout(2)
            s.connect((host, port))
            print(f'Connected to {host}:{port} successfully.')
    except ConnectionRefusedError:
        print(f'ConnectionRefusedError: Nothing listening on {host}:{port}')
    except socket.timeout:
        print(f'socket.timeout: Took too long connecting to {host}:{port}')
    except socket.gaierror as e:
        print(f'socket.gaierror: Cannot resolve hostname — {e}')
    except OSError as e:
        print(f'OSError (fallback): {e}')

# Nothing is running on port 29999 — expect ConnectionRefusedError
safe_connect('localhost', 29999)

# Invalid hostname — expect gaierror
safe_connect('not-a-real-host.invalid', 80)


In [ ]:
# ── 4.4 Echo Client + Background Echo Server ─────────────────────
# Start a minimal echo server in a background thread, then connect.

DEMO_PORT_ECHO = 20200

def _echo_server_once():
    """Receive one message, echo it back, then stop."""
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as srv:
        srv.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
        srv.bind(('localhost', DEMO_PORT_ECHO))
        srv.listen(1)
        srv.settimeout(10)
        try:
            client_sock, _ = srv.accept()
            with client_sock:
                while True:
                    data = client_sock.recv(1024)
                    if not data:
                        break
                    client_sock.sendall(data)  # echo back
        except socket.timeout:
            pass

threading.Thread(target=_echo_server_once, daemon=True).start()
time.sleep(0.1)

# Now the echo client
message = 'Hello from the Jupyter notebook!'
with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
    sock.settimeout(5)
    sock.connect(('localhost', DEMO_PORT_ECHO))
    sock.sendall(message.encode('utf-8'))
    echo = sock.recv(1024)
    print(f'Sent   : {message!r}')
    print(f'Echoed : {echo.decode("utf-8")!r}')
    print(f'Match  : {message == echo.decode("utf-8")}')


<a id='s5'></a>
---
## Section 5 — Writing the Server Program

### The Server Template

```python
import socket

HOST, PORT = 'localhost', 65432

with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as srv:
    srv.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)  # MUST be before bind()
    srv.bind((HOST, PORT))
    srv.listen(5)
    print(f'Listening on {HOST}:{PORT}')
    while True:                                      # loop forever
        client_sock, addr = srv.accept()             # blocks here
        with client_sock:
            while True:
                data = client_sock.recv(1024)
                if not data:   # client disconnected
                    break
                client_sock.sendall(data)
```

### Key Rules for the Server

1. **`SO_REUSEADDR`** — set this BEFORE `bind()` or you get `OSError: Address already in use` on restart.
2. **`while True:`** — the outer loop keeps accepting new clients. Without it, the server exits after one.
3. **`if not data: break`** — `recv()` returns empty bytes `b''` when the client disconnects. Always check for this.


In [ ]:
# ── 5.1 SO_REUSEADDR Demonstration ──────────────────────────────
# Show what happens when you bind WITHOUT SO_REUSEADDR after a server
# that just closed the port.

# Bind and immediately release a port
PORT_TEST = 20300

s1 = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
s1.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
s1.bind(('localhost', PORT_TEST))
s1.close()   # close immediately (simulates server restart)

# Now bind AGAIN on the same port WITH SO_REUSEADDR
s2 = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
s2.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
s2.bind(('localhost', PORT_TEST))
s2.close()
print(f'Rebound to port {PORT_TEST} immediately after closing — SO_REUSEADDR works!')


In [ ]:
# ── 5.2 The Empty Data Signal ────────────────────────────────────
# recv() returns b'' when the remote side closes the connection.
# b'' is falsy in Python — so 'if not data: break' detects disconnect.

print('Testing falsy / truthy bytes values:')
print(f'  bool(b"")       = {bool(b"")}   <-- FALSY (client disconnected)')
print(f'  bool(b"Hello") = {bool(b"Hello")} <-- TRUTHY (data received)')
print(f'  not b""        = {not b""}    <-- True means BREAK')
print(f'  not b"Hello"   = {not b"Hello"}  <-- False means CONTINUE')

print()
print('Server inner loop pattern:')
print('  while True:')
print('      data = client_socket.recv(1024)')
print('      if not data:   # <-- catches b""')
print('          break      # <-- client disconnected')
print('      client_socket.sendall(data)')


In [ ]:
# ── 5.3 Iterative Echo Server — Full Demo ────────────────────────
# This server handles ONE client at a time (iterative / sequential).
# The server runs in a background thread so the notebook stays interactive.

DEMO_PORT_ITER = 20400
_iter_server_running = True

def _iterative_echo_server():
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as srv:
        srv.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
        srv.bind(('localhost', DEMO_PORT_ITER))
        srv.listen(5)
        srv.settimeout(15)   # auto-stop after 15 s idle
        print(f'[Server] Iterative echo server on port {DEMO_PORT_ITER}')
        try:
            while True:
                try:
                    client_sock, addr = srv.accept()
                except socket.timeout:
                    break   # no new clients for 15 s — stop
                print(f'[Server] Client connected: {addr}')
                with client_sock:
                    while True:
                        data = client_sock.recv(1024)
                        if not data:
                            break
                        client_sock.sendall(data)
                print(f'[Server] Client {addr} disconnected.')
        except Exception as e:
            print(f'[Server] Stopped: {e}')
    print('[Server] Shut down.')

threading.Thread(target=_iterative_echo_server, daemon=True).start()
time.sleep(0.15)
print('Server started. Run the next cell to connect clients.')


In [ ]:
# ── 5.4 Connecting to the Iterative Server ───────────────────────
# Send three different messages and receive the echoes.

messages = [
    'First message',
    'Second message — a bit longer',
    'Third message!',
]

with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
    sock.settimeout(5)
    sock.connect(('localhost', DEMO_PORT_ITER))
    for msg in messages:
        sock.sendall(msg.encode('utf-8'))
        reply = sock.recv(1024).decode('utf-8')
        print(f'Sent: {msg!r:<40}  Echo: {reply!r}')
print('Done.')


### Threaded Server — One Thread Per Client

The iterative server handles **one client at a time**. While serving client A, clients B and C must wait. A **threaded server** spawns a new thread for each client so they are all served in parallel.

**Key threading rules:**

| Rule | Reason |
|---|---|
| `target=handle_client` | The function to run in the thread |
| `args=(client_sock, addr)` | Pass the client socket to the thread |
| `daemon=True` | Thread exits automatically when main program exits |
| `.start()` | Launch the thread immediately |

```python
def handle_client(client_socket, addr):
    with client_socket:
        while True:
            data = client_socket.recv(1024)
            if not data:
                break
            client_socket.sendall(data)

# In the server loop:
while True:
    client_sock, addr = srv.accept()
    threading.Thread(
        target=handle_client,
        args=(client_sock, addr),
        daemon=True
    ).start()
```


In [ ]:
# ── 5.5 Threaded Echo Server — Demo ─────────────────────────────
# Starts a threaded echo server, then connects THREE clients
# simultaneously to show they are all served in parallel.

DEMO_PORT_THREAD = 20500

def _handle_client(client_socket, addr):
    with client_socket:
        while True:
            data = client_socket.recv(1024)
            if not data:
                break
            # Echo back with a tag showing which thread handled it
            response = f'[{threading.current_thread().name}] {data.decode()}'
            client_socket.sendall(response.encode('utf-8'))

def _threaded_echo_server():
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as srv:
        srv.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
        srv.bind(('localhost', DEMO_PORT_THREAD))
        srv.listen(10)
        srv.settimeout(15)
        try:
            while True:
                try:
                    cs, addr = srv.accept()
                except socket.timeout:
                    break
                threading.Thread(
                    target=_handle_client,
                    args=(cs, addr),
                    daemon=True,
                    name=f'ClientThread-{addr[1]}'
                ).start()
        except Exception:
            pass

threading.Thread(target=_threaded_echo_server, daemon=True).start()
time.sleep(0.15)

# Connect 3 clients simultaneously
results = []

def _client_task(msg):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.settimeout(5)
        s.connect(('localhost', DEMO_PORT_THREAD))
        s.sendall(msg.encode())
        results.append(s.recv(1024).decode())

threads = [
    threading.Thread(target=_client_task, args=(f'Client {i} says hello',))
    for i in range(1, 4)
]
for t in threads:
    t.start()
for t in threads:
    t.join(timeout=5)

print('Responses from the threaded server:')
for r in results:
    print(' ', r)


<a id='s6'></a>
---
## Section 6 — Library Server Capstone (Sockets + SQLite)

We now combine **Unit 6** (SQLite) with **Unit 7** (sockets) to build a real data-driven server.

### Architecture

```
Library Client                      Library Server
──────────────────                  ────────────────────────────────
search_library('Python')            Receives JSON request
  ↓                                 Queries SQLite database
  Send JSON request                 Returns JSON list of books
  {'action': 'search',              ↓
   'query':  'Python'}         Client decodes and displays results
```

### JSON Protocol

All messages are **JSON strings** — structured, human-readable, easy to parse.

| Request | JSON sent | Response |
|---|---|---|
| Search | `{"action": "search", "query": "Python"}` | JSON list of matching books |
| Add | `{"action": "add", "title": "...", "author": "..."}` | `{"status": "ok"}` |
| List all | `{"action": "list"}` | JSON list of all books |
| Quit | `{"action": "quit"}` | Server closes this connection |

> This is the **same pattern** used by real REST APIs. The next step is just adding HTTP on top.


In [ ]:
# ── 6.1 Library Database Setup ───────────────────────────────────
# Create an in-memory SQLite database with sample books.
# This is shared by the server thread.

import sqlite3

_lib_db_lock = threading.Lock()
_lib_conn = sqlite3.connect(':memory:', check_same_thread=False)

def _init_library():
    with _lib_db_lock:
        cur = _lib_conn.cursor()
        cur.execute("""
            CREATE TABLE IF NOT EXISTS books (
                id     INTEGER PRIMARY KEY AUTOINCREMENT,
                title  TEXT NOT NULL,
                author TEXT NOT NULL
            )
        """)
        sample_books = [
            ('Learning Python', 'Mark Lutz'),
            ('Fluent Python', 'Luciano Ramalho'),
            ('Python Cookbook', 'David Beazley'),
            ('Clean Code', 'Robert C. Martin'),
            ('The Pragmatic Programmer', 'Andrew Hunt'),
            ('Network Programming in Python', 'Brandon Rhodes'),
        ]
        cur.executemany(
            'INSERT INTO books (title, author) VALUES (?, ?)', sample_books
        )
        _lib_conn.commit()

_init_library()
print('Library database initialised with sample books.')

# Preview
cur = _lib_conn.cursor()
cur.execute('SELECT id, title, author FROM books')
print(f'{'ID':<4} {'Title':<35} Author')
print('-' * 60)
for row in cur.fetchall():
    print(f'{row[0]:<4} {row[1]:<35} {row[2]}')


In [ ]:
# ── 6.2 Library Server Logic ─────────────────────────────────────

DEMO_PORT_LIB = 20600

def _handle_library_client(client_socket, addr):
    """Handle one library client in its own thread."""
    with client_socket:
        while True:
            raw = client_socket.recv(4096)
            if not raw:
                break
            try:
                request = json.loads(raw.decode('utf-8'))
                action  = request.get('action', '')

                if action == 'quit':
                    client_socket.sendall(
                        json.dumps({'status': 'bye'}).encode()
                    )
                    break

                elif action == 'list':
                    with _lib_db_lock:
                        cur = _lib_conn.cursor()
                        cur.execute('SELECT id, title, author FROM books')
                        books = [{'id': r[0], 'title': r[1], 'author': r[2]}
                                 for r in cur.fetchall()]
                    client_socket.sendall(
                        json.dumps({'books': books}).encode()
                    )

                elif action == 'search':
                    q = f"%{request.get('query', '')}%"
                    with _lib_db_lock:
                        cur = _lib_conn.cursor()
                        cur.execute(
                            'SELECT id, title, author FROM books '
                            'WHERE title LIKE ? OR author LIKE ?',
                            (q, q)
                        )
                        books = [{'id': r[0], 'title': r[1], 'author': r[2]}
                                 for r in cur.fetchall()]
                    client_socket.sendall(
                        json.dumps({'books': books}).encode()
                    )

                elif action == 'add':
                    title  = request.get('title', 'Unknown')
                    author = request.get('author', 'Unknown')
                    with _lib_db_lock:
                        cur = _lib_conn.cursor()
                        cur.execute(
                            'INSERT INTO books (title, author) VALUES (?, ?)',
                            (title, author)
                        )
                        _lib_conn.commit()
                    client_socket.sendall(
                        json.dumps({'status': 'ok', 'added': title}).encode()
                    )

                else:
                    client_socket.sendall(
                        json.dumps({'error': f'Unknown action: {action}'}).encode()
                    )

            except json.JSONDecodeError:
                client_socket.sendall(
                    json.dumps({'error': 'Invalid JSON'}).encode()
                )

def _library_server():
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as srv:
        srv.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
        srv.bind(('localhost', DEMO_PORT_LIB))
        srv.listen(10)
        srv.settimeout(20)
        try:
            while True:
                try:
                    cs, addr = srv.accept()
                except socket.timeout:
                    break
                threading.Thread(
                    target=_handle_library_client,
                    args=(cs, addr),
                    daemon=True
                ).start()
        except Exception:
            pass

threading.Thread(target=_library_server, daemon=True).start()
time.sleep(0.15)
print(f'Library server started on port {DEMO_PORT_LIB}')


In [ ]:
# ── 6.3 Library Client — Helper ──────────────────────────────────

def library_request(action, **kwargs):
    """Send a JSON request to the library server and return the response."""
    payload = {'action': action, **kwargs}
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.settimeout(5)
        s.connect(('localhost', DEMO_PORT_LIB))
        s.sendall(json.dumps(payload).encode('utf-8'))
        data = s.recv(8192)
    return json.loads(data.decode('utf-8'))

print('library_request() helper ready.')


In [ ]:
# ── 6.4 List All Books ───────────────────────────────────────────
response = library_request('list')
print(f"Total books: {len(response['books'])}")
print()
print(f"{'ID':<4} {'Title':<38} Author")
print('-' * 65)
for b in response['books']:
    print(f"{b['id']:<4} {b['title']:<38} {b['author']}")


In [ ]:
# ── 6.5 Search for Books ─────────────────────────────────────────
response = library_request('search', query='Python')
print(f"Search results for 'Python': {len(response['books'])} book(s)")
for b in response['books']:
    print(f"  [{b['id']}] {b['title']} — {b['author']}")

print()
response2 = library_request('search', query='Robert')
print(f"Search results for 'Robert': {len(response2['books'])} book(s)")
for b in response2['books']:
    print(f"  [{b['id']}] {b['title']} — {b['author']}")


In [ ]:
# ── 6.6 Add a New Book ───────────────────────────────────────────
response = library_request('add',
    title='Automate the Boring Stuff with Python',
    author='Al Sweigart')
print(f"Add response: {response}")

# Verify it was added
response = library_request('list')
print(f"\nTotal books now: {len(response['books'])}")
print(f"Last entry: {response['books'][-1]}")


<a id='practice'></a>
---
## Practice Tasks

Work through these tasks using the concepts from Sections 1–6.
Each task has a clear requirement and a hint.


### Practice Task 1 — Encode/Decode Round-Trip

**Task:** Write a function `encode_decode(text)` that:
1. Encodes the given string to bytes using UTF-8
2. Prints the bytes object
3. Decodes the bytes back to a string
4. Returns the decoded string

Then call it with: `'Network programming is fun!'`

> **Hint:** Use `.encode('utf-8')` and `.decode('utf-8')`


In [ ]:
# Practice Task 1 — your solution here
def encode_decode(text):
    # Step 1: encode to bytes
    # Step 2: print the bytes object
    # Step 3: decode back
    # Step 4: return decoded
    pass

result = encode_decode('Network programming is fun!')
print(f'Result: {result!r}')


### Practice Task 2 — Error-Safe Client

**Task:** Write a function `safe_fetch(host, port, message)` that:
1. Creates a TCP socket with a 3-second timeout
2. Connects to `(host, port)`
3. Sends `message` encoded as UTF-8
4. Returns the decoded response string
5. Catches `ConnectionRefusedError` and returns `'Server not available'`
6. Catches `socket.timeout` and returns `'Connection timed out'`

Test it by calling `safe_fetch('localhost', 29999, 'hello')` — port 29999 is not in use, so you should get the `ConnectionRefusedError` message.

> **Hint:** Use `try / except` with a `with socket.socket(...) as s:` block.


In [ ]:
# Practice Task 2 — your solution here
def safe_fetch(host, port, message):
    pass

# Test 1 — nothing listening on this port
print(safe_fetch('localhost', 29999, 'hello'))

# Test 2 — connect to the echo server from Section 4 (if still running)
# print(safe_fetch('localhost', DEMO_PORT_ECHO, 'test'))


### Practice Task 3 — One-Shot Server

**Task:** Write a function `start_greeting_server(port)` that:
1. Creates a socket, binds to `localhost:port`, listens with backlog 1
2. Accepts **one** client connection
3. Reads the client's name (up to 256 bytes, decoded as UTF-8)
4. Replies with `f'Hello, {name}! Welcome to network programming.'` encoded as UTF-8
5. Closes the client connection
6. Closes the server socket and returns

Run the server in a background thread, then connect a client that sends `b'Alice'` and print the greeting.

> **Hint:** Use `threading.Thread(target=..., daemon=True).start()` then `time.sleep(0.1)` before connecting.


In [ ]:
# Practice Task 3 — your solution here
PRACTICE_PORT_3 = 20700

def start_greeting_server(port):
    pass

# Start server in background thread
threading.Thread(
    target=start_greeting_server,
    args=(PRACTICE_PORT_3,),
    daemon=True
).start()
time.sleep(0.1)

# Client — sends name, receives greeting
with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
    s.settimeout(5)
    s.connect(('localhost', PRACTICE_PORT_3))
    s.sendall(b'Alice')
    greeting = s.recv(256).decode('utf-8')
    print(greeting)


<a id='debug'></a>
---
## Debugging Exercises

Each cell below contains **broken code**. Identify the bug, fix it, and explain what the bug was.


### Debug 1 — Wrong Data Type

The code below raises a `TypeError`. Find and fix the bug.


In [ ]:
# Debug 1 — find and fix the bug
# Hint: what type does sendall() expect?

def buggy_client_1():
    DEMO_PORT_ECHO = 20200  # echo server from Section 4
    message = 'Hello, server!'   # <-- this is the bug
    try:
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            s.settimeout(5)
            s.connect(('localhost', DEMO_PORT_ECHO))
            s.sendall(message)   # BUG: should be message.encode('utf-8')
            print(s.recv(1024))
    except TypeError as e:
        print(f'TypeError caught: {e}')
        print('Fix: encode the string before sending')

buggy_client_1()

# YOUR FIX:
# Replace: s.sendall(message)
# With:    s.sendall(message.encode('utf-8'))


### Debug 2 — Forgetting to Decode the Response

The code prints bytes instead of a string. Fix it.


In [ ]:
# Debug 2 — find and fix the bug
# Start a quick echo server
DEMO_PORT_D2 = 20800

def _echo_once_d2():
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as srv:
        srv.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
        srv.bind(('localhost', DEMO_PORT_D2))
        srv.listen(1)
        srv.settimeout(10)
        try:
            cs, _ = srv.accept()
            with cs:
                data = cs.recv(1024)
                cs.sendall(data)
        except socket.timeout:
            pass

threading.Thread(target=_echo_once_d2, daemon=True).start()
time.sleep(0.1)

with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
    s.settimeout(5)
    s.connect(('localhost', DEMO_PORT_D2))
    s.sendall(b'Hello!')
    response = s.recv(1024)
    print(f'Response: {response}')   # BUG: prints bytes, not string
    # FIX: print(f'Response: {response.decode("utf-8")}')


### Debug 3 — Missing `if not data: break`

The server below will loop forever after the client disconnects, consuming 100% CPU. Add the missing check.


In [ ]:
# Debug 3 — identify the missing pattern
# This is a CODE REVIEW exercise — do not run this server cell.
# Instead, identify the bug and write the corrected version below.

BUGGY_SERVER_CODE = """
import socket
HOST, PORT = 'localhost', 65432
with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as srv:
    srv.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
    srv.bind((HOST, PORT))
    srv.listen(5)
    while True:
        client_sock, addr = srv.accept()
        with client_sock:
            while True:
                data = client_sock.recv(1024)
                # BUG: missing 'if not data: break'
                client_sock.sendall(data)  # loops forever on b''
"""

print('Bug identified: missing "if not data: break" check.')
print()
print('Corrected inner loop:')
print('    while True:')
print('        data = client_sock.recv(1024)')
print('        if not data:   # client disconnected')
print('            break')
print('        client_sock.sendall(data)')


<a id='challenges'></a>
---
## Mini Coding Challenges

Larger tasks that combine multiple concepts. These are good exam preparation.


### Challenge 1 — Uppercase Echo Server

**Task:** Build a server that:
- Listens on `localhost:20900`
- For each message received, replies with the **UPPERCASE** version
- Handles multiple messages per connection
- Disconnects gracefully when the client sends `b'QUIT'`

Then write a client that:
- Connects and sends three test messages: `'hello'`, `'world'`, `'quit'`
  (Note: `'quit'` should NOT trigger the QUIT protocol — only `'QUIT'` does)
- Sends `'QUIT'` to close the connection cleanly
- Prints each response

> **Hint:** Use `.upper()` on the decoded message before re-encoding and sending.


In [ ]:
# Challenge 1 — Uppercase Echo Server
CHALLENGE_PORT_1 = 20900

# ── Server (in background thread) ──
def _uppercase_server():
    # TODO: implement the uppercase echo server
    pass

threading.Thread(target=_uppercase_server, daemon=True).start()
time.sleep(0.1)

# ── Client ──
messages = ['hello', 'world', 'quit', 'QUIT']   # 'QUIT' ends the session

with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
    s.settimeout(5)
    s.connect(('localhost', CHALLENGE_PORT_1))
    for msg in messages:
        s.sendall(msg.encode())
        response = s.recv(1024)
        if not response:
            break
        print(f'Sent: {msg!r:<10}  Received: {response.decode()!r}')


### Challenge 2 — Word Count Server

**Task:** Build a server that:
- Listens on `localhost:21000`
- Receives a sentence as bytes
- Replies with a JSON object:
  `{"words": <count>, "chars": <count>, "upper": "<UPPERCASE_SENTENCE>"}`
- Handles multiple requests per connection
- Closes when client sends `b'QUIT'`

Test with sentences:
- `'The quick brown fox'`
- `'Network programming with Python'`

> **Hint:** Use `json.dumps()` to create the response and `json.loads()` in the client to parse it.


In [ ]:
# Challenge 2 — Word Count Server
CHALLENGE_PORT_2 = 21000

# ── Server ──
def _word_count_server():
    # TODO: implement the word count server
    pass

threading.Thread(target=_word_count_server, daemon=True).start()
time.sleep(0.1)

# ── Client ──
test_sentences = [
    'The quick brown fox',
    'Network programming with Python',
]

with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
    s.settimeout(5)
    s.connect(('localhost', CHALLENGE_PORT_2))
    for sentence in test_sentences:
        s.sendall(sentence.encode())
        data = s.recv(1024)
        result = json.loads(data.decode())
        print(f'Sentence: {sentence!r}')
        print(f'  Words : {result["words"]}')
        print(f'  Chars : {result["chars"]}')
        print(f'  Upper : {result["upper"]}')
        print()
    s.sendall(b'QUIT')


<a id='summary'></a>
---
## Summary & Checklist

### The Five Rules of Network Programming

| # | Rule |
|---|---|
| 1 | **Always use `with`** — sockets must be closed; `with` guarantees cleanup |
| 2 | **Always encode before sending** — `'text'.encode('utf-8')` |
| 3 | **Always decode after receiving** — `data.decode('utf-8')` |
| 4 | **Always use `sendall()`** — `send()` may only send part of your data |
| 5 | **Always check `if not data: break`** — empty recv = client disconnected |

### Key Concepts Covered

| Concept | Where Covered |
|---|---|
| Socket creation — `AF_INET`, `SOCK_STREAM` | Section 1 |
| The six-step server pattern | Section 2 |
| TCP three-way handshake | Section 3 |
| Bytes vs strings — encode / decode | Section 4 |
| `SO_REUSEADDR` | Section 5 |
| `if not data: break` | Section 5 |
| Threaded server — `daemon=True` | Section 5 |
| JSON protocol over sockets | Section 6 |
| Sockets + SQLite capstone | Section 6 |

### Self-Check Checklist

Tick each item when you are confident:

- [ ] I can explain what `socket.AF_INET` and `socket.SOCK_STREAM` mean
- [ ] I know the difference between TCP and UDP
- [ ] I can write the six-step server pattern from memory
- [ ] I know why `SO_REUSEADDR` must be set before `bind()`
- [ ] I understand why `recv()` returns `b''` and what to do about it
- [ ] I can write a client that handles `ConnectionRefusedError`
- [ ] I understand how to use threads to serve multiple clients
- [ ] I can send and receive JSON messages over a socket
- [ ] I have completed all three Practice Tasks
- [ ] I have fixed all three Debugging Exercises


In [ ]:
# ── Final Self-Test ──────────────────────────────────────────────
# Answer these True/False questions to check your understanding.

questions = [
    ('Sockets can send Python strings directly without encoding.',  False),
    ('recv() returns b"" when the client disconnects.',           True),
    ('SO_REUSEADDR must be set BEFORE bind().',                    True),
    ('sendall() may send fewer bytes than requested.',              False),
    ('daemon=True means the thread runs forever in the background.',False),
    ('localhost and 127.0.0.1 refer to the same address.',         True),
    ('The server socket returned by accept() should be used for data.', False),
    ('TCP guarantees data arrives in order; UDP does not.',        True),
]

print(f'{'Statement':<56} {'Answer'}')
print('-' * 65)
for stmt, ans in questions:
    print(f'{stmt:<56} {str(ans)}')

print()
print('Check your answers against the lecture notes and sections above.')


---

## Next Steps

- **Unit 8:** Regular Expressions and Advanced Threading
- **Extend this notebook:** Add a `delete` action to the library server
- **Challenge:** Add a `count` action that returns the total number of books
- **Research:** Look up `ssl.wrap_socket()` to secure your server with TLS

---
*BT151CO — Object-Oriented Programming with Python · Unit 7*
